# 03 — Exploratory Data Analysis

Visualise distributions, correlations, seasonal patterns, and test business hypotheses.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'text.color': '#e2e8f0', 'grid.color': '#334155',
    'figure.titlesize': 16, 'axes.titlesize': 13,
})
PALETTE = ['#38bdf8','#fb7185','#34d399','#fbbf24','#a78bfa','#f97316']

with open('../configs/paths.yaml') as f:
    paths = yaml.safe_load(f)
df = pd.read_csv(f'../{paths["data"]["raw"]}')
df['dteday'] = pd.to_datetime(df['dteday'], dayfirst=True)
print(df.shape)

## 3.1  Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable: cnt (Total Daily Rentals)')
axes[0].hist(df['cnt'], bins=30, color=PALETTE[0], edgecolor='#0f172a', alpha=0.9)
axes[0].set(title='Distribution', xlabel='cnt', ylabel='Frequency')
bp = axes[1].boxplot(df['cnt'], patch_artist=True,
    boxprops=dict(facecolor=PALETTE[0], color='white'),
    medianprops=dict(color=PALETTE[1], linewidth=2),
    whiskerprops=dict(color='white'), capprops=dict(color='white'),
    flierprops=dict(marker='o', markerfacecolor=PALETTE[1], markersize=5))
axes[1].set(title='Box Plot', ylabel='cnt')
axes[1].set_xticks([])
plt.tight_layout()
plt.savefig('../outputs/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.2  Time Series Plot

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df['dteday'], df['cnt'], color=PALETTE[0], linewidth=1.2, alpha=0.9)
ax.fill_between(df['dteday'], df['cnt'], alpha=0.15, color=PALETTE[0])
rolling = df.set_index('dteday')['cnt'].rolling(30).mean()
ax.plot(rolling.index, rolling.values, color=PALETTE[1], linewidth=2, label='30-day avg')
ax.set(title='Daily Bike Rentals Over Time', xlabel='Date', ylabel='cnt')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/time_series.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3  Seasonal & Calendar Patterns

In [ ]:
df['season_label'] = df['season'].map({1:'Spring',2:'Summer',3:'Fall',4:'Winter'})
df['weather_label'] = df['weathersit'].map({1:'Clear',2:'Mist',3:'Light Rain',4:'Heavy Rain'})
df['weekday_label'] = df['weekday'].map({0:'Sun',1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat'})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('cnt by Calendar & Weather Features')

sns.boxplot(data=df, x='season_label', y='cnt',
    order=['Spring','Summer','Fall','Winter'], palette=PALETTE[:4], ax=axes[0,0])
axes[0,0].set_title('By Season'); axes[0,0].set_xlabel('')

month_avg = df.groupby('mnth')['cnt'].mean()
axes[0,1].bar(month_avg.index, month_avg.values, color=PALETTE[0], edgecolor='#0f172a')
axes[0,1].set(title='Average by Month', xlabel='Month'); axes[0,1].set_xticks(range(1,13))

sns.boxplot(data=df, x='yr', y='cnt', palette=[PALETTE[0],PALETTE[1]], ax=axes[0,2])
axes[0,2].set_xticklabels(['2018','2019']); axes[0,2].set(title='Year Comparison', xlabel='')

sns.boxplot(data=df, x='weekday_label', y='cnt',
    order=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], palette=PALETTE, ax=axes[1,0])
axes[1,0].set(title='By Weekday', xlabel='')

present = [w for w in ['Clear','Mist','Light Rain','Heavy Rain'] if w in df['weather_label'].values]
sns.boxplot(data=df, x='weather_label', y='cnt', order=present,
    palette=PALETTE[:len(present)], ax=axes[1,1])
axes[1,1].set(title='By Weather', xlabel='')

dt = df.copy()
dt['day_type'] = 'Weekday'
dt.loc[dt['holiday']==1,'day_type'] = 'Holiday'
dt.loc[(dt['workingday']==0)&(dt['holiday']==0),'day_type'] = 'Weekend'
sns.boxplot(data=dt, x='day_type', y='cnt',
    order=['Weekday','Weekend','Holiday'], palette=PALETTE[:3], ax=axes[1,2])
axes[1,2].set(title='Day Type', xlabel='')

for ax in axes.flat:
    ax.set_ylabel('cnt'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/calendar_weather_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4  Numeric Feature Distributions

In [ ]:
num_cols = ['temp','atemp','hum','windspeed','cnt']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Numeric Feature Distributions')
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=25, color=PALETTE[i%len(PALETTE)], edgecolor='#0f172a', alpha=0.9)
    axes[i].set(title=col, xlabel=col)
    axes[i].set_ylabel('Frequency' if i==0 else '')
    axes[i].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5  Correlation Heatmap

In [ ]:
corr_df = df.drop(columns=['instant','dteday','casual','registered',
                            'season_label','weather_label','weekday_label'])
corr = corr_df.corr()
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, linecolor='#0f172a',
            annot_kws={'size': 9}, ax=ax)
ax.set_title('Correlation Matrix (lower triangle)')
plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelations with cnt:')
print(corr['cnt'].drop('cnt').sort_values(ascending=False).to_string())

## 3.6  Scatter: Numeric Features vs cnt

In [ ]:
num_feats = ['temp','atemp','hum','windspeed']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Numeric Features vs cnt')
for i, col in enumerate(num_feats):
    axes[i].scatter(df[col], df['cnt'], alpha=0.4, color=PALETTE[i], s=20)
    z = np.polyfit(df[col], df['cnt'], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), color='white', linewidth=2)
    axes[i].set(title=col, xlabel=col)
    axes[i].set_ylabel('cnt' if i==0 else '')
    axes[i].grid(alpha=0.2)
plt.tight_layout()
plt.savefig('../outputs/scatter_numeric_vs_cnt.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7  Outlier Inspection

In [ ]:
def detect_outliers_iqr(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    return ((s < lo) | (s > hi)).sum(), lo, hi

print(f'{"Column":<15} {"# Outliers":>12} {"Lower":>10} {"Upper":>10}')
print('-'*50)
for col in ['temp','atemp','hum','windspeed','cnt']:
    n, lo, hi = detect_outliers_iqr(df[col])
    print(f'{col:<15} {n:>12} {lo:>10.3f} {hi:>10.3f}')

## EDA Summary

| Hypothesis | Finding |
|---|---|
| Temperature drives demand | ✅ Strong positive correlation |
| Bad weather suppresses demand | ✅ weathersit 3/4 → lower counts |
| Working days → stable commuters | ✅ Less variance on working days |
| 2019 > 2018 | ✅ Higher median in yr=1 |
| Holidays suppress demand | ✅ Slight dip vs weekdays |

**Next:** `04_Data_Preprocessing.ipynb`
